# 04 - Train the v3 No-KD multimodal baseline

Trains only the fusion head on hard labels using the freshly reproduced v3 text/audio branch embeddings.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, shutil, subprocess, base64

DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
EXP_ROOT = DAIC / 'experiments' / 'exact_v3_reproduction'
ACTIVE = EXP_ROOT / 'ACTIVE_RUN.txt'
assert ACTIVE.is_file(), 'Run notebooks/00_start_new_run.ipynb first.'
RUN = EXP_ROOT / ACTIVE.read_text().strip()
assert RUN.is_dir(), RUN

REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
CORE_COMMIT = 'd6ed3a8b54ec9cfa7c706cb441671c6e8b07096d'
CODE = Path('/content/reliability-aware-depression-kd-exact-v3')

token = userdata.get('GITHUB_TOKEN')
assert token, 'Add GITHUB_TOKEN to Colab Secrets.'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy()
env['GIT_CONFIG_COUNT']='1'
env['GIT_CONFIG_KEY_0']='http.https://github.com/.extraheader'
env['GIT_CONFIG_VALUE_0']=f'AUTHORIZATION: basic {auth}'

if CODE.exists(): shutil.rmtree(CODE)
CODE.mkdir(parents=True)
subprocess.run(['git','-C',str(CODE),'init'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'remote','add','origin',f'https://github.com/{REPO}.git'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',CORE_COMMIT],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'checkout','--detach','FETCH_HEAD'],check=True,env=env)
head=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
assert head==CORE_COMMIT,(head,CORE_COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
print('RUN:',RUN)
print('Pinned core:',head)


In [ ]:
TEXT=RUN/'03_student_text'
AUDIO=RUN/'03_student_audio'
OUT=RUN/'04_no_kd'
assert TEXT.is_dir() and AUDIO.is_dir(),'Run notebook 03 first.'

subprocess.run([sys.executable,'-m','scripts.students.participant_student_v3.train_fusion_no_kd',
 '--text-branch',str(TEXT),'--audio-branch',str(AUDIO),'--output',str(OUT),
 '--seed','103','--epochs','80','--patience','20','--batch-size','8',
 '--lr','0.0002','--weight-decay','0.0005','--dropout','0.35'],cwd=CODE,check=True)

d=json.loads((OUT/'metrics.json').read_text()); m=d['dev34']
print(json.dumps(m,indent=2))
assert d['protocol']['test_opened'] is False
assert round(float(m['macro_f1']),4)==0.7043
assert round(float(m['depressed_f1']),4)==0.6087
assert round(float(m['auroc']),4)==0.7470
print('PASS: No-KD DEV-34 reproduced.')